In [13]:
import numpy as np
import matplotlib.pyplot
import random
import math
from scipy.optimize import minimize

In [14]:
# Defining parameters
a_min = 0
a_max = 2
Num_of_a = 30       # number of 'a' values to sample (like 30 radii in the circle problem)
t_max = 1
Num_of_steps = 50   # number of time steps - bigger = smoother Euler
M, N = 3, 3         # polynomial degrees in a and t - START SMALL (recovers 4 terms)


In [36]:
# Cell 2 - Generate data with Euler's method

# Step 1: sample the 'a' values (uniform, like random_radii in the circle code)
#   -> Num_of_a values between a_min and a_max
# Why do we need a values?
a_values = np.random.uniform(a_min, a_max, Num_of_a)

# Step 2: build the time grid
#   -> time points from 0 to t_max, Num_of_steps of them
time_grid = np.linspace(0, t_max, Num_of_steps)

# Step 3: storage for the flattened triples
# TODO: Why do we need these lists?
a_data = []
t_data = []
x_data = []

# Step 4: loop over each 'a'
#   for each a:
#       start x at 1.0   (initial condition x(0)=1)
#       loop over the time grid:
#           record current (a, t, x) into the three lists
#           Euler update: x = x + dt * a * x
for a in a_values:
    x = 1.0             # initial condition x(0) = 1
    for t in time_grid:
        a_data.append(a)
        t_data.append(t)
        x_data.append(x)
        # Euler update
        x = x + dt * a * x

In [16]:
print(len(a_data), len(t_data), len(x_data))
print(a_data[0], t_data[0], x_data[0])

1500 1500 1500
1.8830447435795388 0.0 1.0


In [17]:
# Cell 3 - Build the bivariate design matrix D

a_arr = np.array(a_data)    # all growth rates, shape (1500,)
t_arr = np.array(t_data)    # all times,        shape (1500,)
b     = np.array(x_data)    # target outputs,   shape (1500,)

columns = []
for m in range(M + 1):           # powers of a: 0,1,2,3
    for n in range(N + 1):       # powers of t: 0,1,2,3
        col = a_arr ** m * t_arr ** n                # <-- the one blank
        columns.append(col)

D = np.column_stack(columns)
print("D shape:", D.shape)

D shape: (1500, 16)


In [18]:
# Cell 4 - Least Squares (the "L" in LSSR)
c = np.linalg.lstsq(D, b, rcond=None)[0]

print(f"{'(m,n)':<8}{'col':<6}{'value':>10}{'target':>10}")
print("-" * 34)
for m in range(M + 1):
    for n in range(N + 1):
        idx = m * (N + 1) + n
        target = 1 / math.factorial(n) if m == n else 0.0
        star = "  <-- diagonal" if m == n else ""
        print(f"({m},{n})   {idx:<6}{c[idx]:>10.4f}{target:>10.4f}{star}")

(m,n)   col        value    target
----------------------------------
(0,0)   0         1.0004    1.0000  <-- diagonal
(0,1)   1        -0.0084    0.0000
(0,2)   2         0.0341    0.0000
(0,3)   3        -0.0402    0.0000
(1,0)   4        -0.0045    0.0000
(1,1)   5         1.0725    1.0000  <-- diagonal
(1,2)   6        -0.3761    0.0000
(1,3)   7         0.4469    0.0000
(2,0)   8         0.0093    0.0000
(2,1)   9        -0.1992    0.0000
(2,2)   10        1.2554    0.5000  <-- diagonal
(2,3)   11       -0.9321    0.0000
(3,0)   12       -0.0057    0.0000
(3,1)   13        0.1173    0.0000
(3,2)   14       -0.4961    0.0000
(3,3)   15        0.7586    0.1667  <-- diagonal


In [24]:
# Cell 5 - Sparsify (hard thresholding, the "S" in LSSR)
c_sparse = c.copy()                      # work on a copy, keep original c intact

threshold = np.max(np.abs(c)) / 10.0  # 1/10 of the biggest magnitude

# zero out everything below threshold
for i in range(len(c_sparse)):
    if abs(c_sparse[i]) < threshold:
        c_sparse[i] = 0.0                # what value kills a coefficient?

# show what survived
print(f"threshold = {threshold:.4f}\n")
survivors = []
for m in range(M + 1):
    for n in range(N + 1):
        idx = m * (N + 1) + n
        if c_sparse[idx] != 0:
            survivors.append((m, n, idx, c_sparse[idx]))
            print(f"  KEPT  a^{m}*t^{n} (col {idx}) = {c_sparse[idx]:.4f}")

print(f"\n{len(survivors)} terms survived out of 16")

threshold = 0.1255

  KEPT  a^0*t^0 (col 0) = 1.0004
  KEPT  a^1*t^1 (col 5) = 1.0725
  KEPT  a^1*t^2 (col 6) = -0.3761
  KEPT  a^1*t^3 (col 7) = 0.4469
  KEPT  a^2*t^1 (col 9) = -0.1992
  KEPT  a^2*t^2 (col 10) = 1.2554
  KEPT  a^2*t^3 (col 11) = -0.9321
  KEPT  a^3*t^2 (col 14) = -0.4961
  KEPT  a^3*t^3 (col 15) = 0.7586

9 terms survived out of 16


In [28]:
# Cell 6 - Refine (the "R" in LSSR)

# 1. Find which columns survived (nonzero in c_sparse)
keep_idx = [i for i in range(len(c_sparse)) if c_sparse[i] != 0]
print("Keeping columns:", keep_idx)

# 2. Build a smaller matrix with ONLY those columns
D_refine = D[:, keep_idx]        # all rows, only the surviving columns
print("D_refine shape:", D_refine.shape)   # (1500, 9)

# 3. Least squares again on the smaller matrix
c_refine_small = np.linalg.lstsq(D_refine, b, rcond=None)[0]

# 4. Put the refined values back into a full 16-length vector
c_refine = np.zeros(len(c))
for slot, idx in enumerate(keep_idx):
    c_refine[idx] = c_refine_small[slot]

# 5. Compare to target
print(f"\n{'(m,n)':<8}{'col':<6}{'refined':>10}{'target':>10}")
print("-" * 34)
for m in range(M + 1):
    for n in range(N + 1):
        idx = m * (N + 1) + n
        if c_refine[idx] != 0:
            target = 1/math.factorial(n) if m == n else 0.0
            star = "  <-- diag" if m == n else ""
            print(f"({m},{n})   {idx:<6}{c_refine[idx]:>10.4f}{target:>10.4f}{star}")

Keeping columns: [0, 5, 6, 7, 9, 10, 11, 14, 15]
D_refine shape: (1500, 9)

(m,n)   col      refined    target
----------------------------------
(0,0)   0         0.9971    1.0000  <-- diag
(1,1)   5         0.9468    1.0000  <-- diag
(1,2)   6         0.0879    0.0000
(1,3)   7         0.0500    0.0000
(2,1)   9         0.0607    0.0000
(2,2)   10        0.4251    0.5000  <-- diag
(2,3)   11       -0.2878    0.0000
(3,2)   14       -0.1522    0.0000
(3,3)   15        0.5049    0.1667  <-- diag


In [35]:
# Cell 7 - ISTA Implementation (Iterative Shrinkage-Thresholding Algorithm)
# Implements the lasso solve from ISTANotes.pdf, Sections 14-15

# --- Soft-thresholding operator, S_eta(z) = sign(z) * max(|z| - eta, 0) ---
def soft_threshold(z, eta):
    return np.sign(z) * np.maximum(np.abs(z) - eta, 0)   # blank 1

# --- Step size: t = 1/L where L = ||D||^2 = sigma_max(D)^2  (Section 15.2) ---
L = np.linalg.norm(D, 2) ** 2     # spectral norm squared = largest singular value^2
t = 1/L                            # blank 2: the safe step size from the notes

# --- Lambda: fraction of lambda_max (Section 16.7) ---
lambda_max = np.max(np.abs(D.T @ b))   # smallest lambda that kills everything
lam = 0.1 * lambda_max                  # 0.1 * lambda_max, like the MATLAB code

# --- ISTA loop (Section 15.1 pseudocode) ---
x = np.zeros(D.shape[1])           # initial guess x^0 = 0  (16 zeros)
maxit = 5000
for k in range(maxit):
    grad = D.T @ (D @ x - b)       # gradient of f:  A^T(Ax - b)
    x = soft_threshold(x - t * grad, t*lam)   # blank 3: the threshold is t*lambda

c_ista = x

# --- Report ---
print(f"L = {L:.4f},  t = 1/L = {t:.6f},  lambda = {lam:.4f}\n")
print(f"{'(m,n)':<8}{'col':<6}{'ISTA':>10}{'target':>10}")
print("-" * 34)
for m in range(M + 1):
    for n in range(N + 1):
        idx = m * (N + 1) + n
        if abs(c_ista[idx]) > 1e-6:
            target = 1/math.factorial(n) if m == n else 0.0
            star = "  <-- diag" if m == n else ""
            print(f"({m},{n})   {idx:<6}{c_ista[idx]:>10.4f}{target:>10.4f}{star}")

L = 41301.7609,  t = 1/L = 0.000024,  lambda = 919.5966

(m,n)   col         ISTA    target
----------------------------------
(0,0)   0         0.3840    1.0000  <-- diag
(3,0)   12        0.1218    0.0000
(3,1)   13        0.4986    0.0000


In [31]:
# Cell 7 - ISTA with column normalization

def soft_threshold(z, eta):
    return np.sign(z) * np.maximum(np.abs(z) - eta, 0)

# --- Normalize columns so ISTA treats them fairly ---
col_norms = np.linalg.norm(D, axis=0)      # norm of each of the 16 columns
D_norm = D / col_norms                       # divide each column by its norm

# --- Step size from the NORMALIZED matrix ---
L = np.linalg.norm(D_norm, 2) ** 2
t = 1 / L

# --- Lambda from the normalized matrix ---
lambda_max = np.max(np.abs(D_norm.T @ b))
lam = 0.1 * lambda_max                        # blank: same fraction as MATLAB (0.1)

# --- ISTA loop on the NORMALIZED matrix ---
x = np.zeros(D_norm.shape[1])
maxit = 5000
for k in range(maxit):
    grad = D_norm.T @ (D_norm @ x - b)        # note: D_norm now, not D
    x = soft_threshold(x - t * grad, t * lam)

# --- Undo the normalization to get REAL coefficients ---
c_ista = x / col_norms                         # convert back to real units

# --- Report ---
print(f"L = {L:.4f},  t = {t:.6f},  lambda = {lam:.4f}\n")
print(f"{'(m,n)':<8}{'col':<6}{'ISTA':>10}{'target':>10}")
print("-" * 34)
for m in range(M + 1):
    for n in range(N + 1):
        idx = m * (N + 1) + n
        if abs(c_ista[idx]) > 1e-6:
            target = 1/math.factorial(n) if m == n else 0.0
            star = "  <-- diag" if m == n else ""
            print(f"({m},{n})   {idx:<6}{c_ista[idx]:>10.4f}{target:>10.4f}{star}")

L = 12.9067,  t = 0.077479,  lambda = 8.1131

(m,n)   col         ISTA    target
----------------------------------
(0,0)   0         0.7847    1.0000  <-- diag
(1,1)   5         1.0992    1.0000  <-- diag
(2,2)   10        0.5869    0.5000  <-- diag
(2,3)   11        0.0122    0.0000


In [33]:
# Cell 7 - ISTA with column normalization

def soft_threshold(z, eta):
    return np.sign(z) * np.maximum(np.abs(z) - eta, 0)

# --- Normalize columns so ISTA treats them fairly ---
col_norms = np.linalg.norm(D, axis=0)      # norm of each of the 16 columns
D_norm = D / col_norms                       # divide each column by its norm

# --- Step size from the NORMALIZED matrix ---
L = np.linalg.norm(D_norm, 2) ** 2
t = 1 / L

# --- Lambda from the normalized matrix ---
lambda_max = np.max(np.abs(D_norm.T @ b))
lam = 0.1 * lambda_max                        # trying a weaker penalty

# --- ISTA loop on the NORMALIZED matrix ---
x = np.zeros(D_norm.shape[1])
maxit = 5000
for k in range(maxit):
    grad = D_norm.T @ (D_norm @ x - b)        # note: D_norm now, not D
    x = soft_threshold(x - t * grad, t * lam)

# --- Undo the normalization to get REAL coefficients ---
c_ista = x / col_norms                         # convert back to real units

# --- Report ---
print(f"L = {L:.4f},  t = {t:.6f},  lambda = {lam:.4f}\n")
print(f"{'(m,n)':<8}{'col':<6}{'ISTA':>10}{'target':>10}")
print("-" * 34)
for m in range(M + 1):
    for n in range(N + 1):
        idx = m * (N + 1) + n
        if abs(c_ista[idx]) > 1e-6:
            target = 1/math.factorial(n) if m == n else 0.0
            star = "  <-- diag" if m == n else ""
            print(f"({m},{n})   {idx:<6}{c_ista[idx]:>10.4f}{target:>10.4f}{star}")

L = 12.9067,  t = 0.077479,  lambda = 8.1131

(m,n)   col         ISTA    target
----------------------------------
(0,0)   0         0.7847    1.0000  <-- diag
(1,1)   5         1.0992    1.0000  <-- diag
(2,2)   10        0.5869    0.5000  <-- diag
(2,3)   11        0.0122    0.0000
